# RetailPulse – Exploratory Data Analysis
Week 1 · Day 1–2 deliverable

In [ ]:
import sys, json
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except ImportError:
    pass

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded ✅')

In [ ]:
# Load datasets
from pathlib import Path

DATA = Path('../data/raw')

sales = pd.read_csv(DATA / 'sales_transactions.csv', parse_dates=['order_date'])
customers = pd.read_csv(DATA / 'customers.csv')
products = pd.read_csv(DATA / 'products_catalog.csv')
inventory = pd.read_csv(DATA / 'inventory.csv')
churn = pd.read_csv(DATA / 'churn_dataset.csv')

print(f'Sales: {sales.shape}')
print(f'Customers: {customers.shape}')
print(f'Products: {products.shape}')
print(f'Inventory: {inventory.shape}')
print(f'Churn dataset: {churn.shape}')

## 1. Sales Overview

In [ ]:
# Daily revenue
daily_rev = sales.groupby('order_date')['revenue'].sum()
daily_rev.plot(title='Daily Revenue', ylabel='Revenue ($)')
plt.tight_layout()
plt.show()
print(f'Total Revenue: ${daily_rev.sum():,.0f}')
print(f'Average Daily: ${daily_rev.mean():,.0f}')

In [ ]:
# Monthly seasonality
sales['month'] = sales['order_date'].dt.month
monthly = sales.groupby('month')['revenue'].sum()
monthly.plot(kind='bar', title='Revenue by Month', ylabel='Revenue ($)', color='steelblue')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Missing values heatmap
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, df) in zip(axes, [('Sales', sales), ('Customers', customers), ('Products', products)]):
    null_pct = df.isnull().mean() * 100
    null_pct[null_pct > 0].plot(kind='bar', ax=ax, color='tomato')
    ax.set_title(f'{name} – Missing %')
    ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 2. Customer Analysis

In [ ]:
# Age distribution
customers['age'].hist(bins=30, color='mediumpurple', edgecolor='white')
plt.title('Customer Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Gender breakdown
customers['gender'].value_counts().plot(kind='pie', autopct='%1.0f%%', title='Gender Split')
plt.tight_layout()
plt.show()

## 3. Churn Analysis

In [ ]:
churn_rate = churn['churn'].mean()
print(f'Overall churn rate: {churn_rate:.1%}')

# Churn by region
if 'region' in churn.columns:
    churn_by_region = churn.groupby('region')['churn'].mean().sort_values()
    churn_by_region.plot(kind='barh', title='Churn Rate by Region', color='coral')
    plt.xlabel('Churn Rate')
    plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    plt.tight_layout()
    plt.show()

## 4. Correlation Heatmap

In [ ]:
num_sales = sales.select_dtypes(include=[np.number])
corr = num_sales.corr()

try:
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
    ax.set_title('Sales Feature Correlation')
    plt.tight_layout()
    plt.show()
except ImportError:
    print(corr.to_string())